# Vector Search: Buiding the Retriever

We move from "I can turn text into vector" to "I can find the most relevant text for a question"

Document A:
Employees receive 30 days of annual leave per calendar year.

Document B:
Employees may work remotely up to three days per week.

Document C:
The company provides occupational health services.

Document D:
Employees can request unpaid leave with manager approval.

We want:
Question
   ↓
embedding
   ↓
compare with document embeddings
   ↓
rank documents
   ↓
return most relevant

Important terminology:
Corpus: Complete collection of documents.
Chunk: A smaller piece of document.
Query: The user's question.
Embeddings: Numerical representation of text.
Similarity: How close two vectors are?
Top k: The k most relevant results.

In [49]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [34]:
# Creating the document
documents = [
    "Employees receive 30 days of annual leave per calendar year.",
    "Employees may work remotely up to three days per week.",
    "The company provides occupational health services.",
    "Employees can request unpaid leave with manager approval.",
    "The company reviews salaries every January."
]

In [35]:
# Embedding the document
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
document_embeddings = model.encode(documents)
document_embeddings.shape

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6662.45it/s]


(5, 384)

In [36]:
# Create the query
query = "How many vacation days do employees receive?"
query_embedding = model.encode([query])

# Calculating the Cosine Similarity
from sklearn.metrics.pairwise import cosine_similarity
scores = cosine_similarity(
    query_embedding,
    document_embeddings
)

# Pairing document with score
results = []
for document, score in zip(documents, scores[0]):
    results.append((document, score))
results


[('Employees receive 30 days of annual leave per calendar year.',
  np.float32(0.7066407)),
 ('Employees may work remotely up to three days per week.',
  np.float32(0.5517914)),
 ('The company provides occupational health services.',
  np.float32(0.24020007)),
 ('Employees can request unpaid leave with manager approval.',
  np.float32(0.32612258)),
 ('The company reviews salaries every January.', np.float32(0.33673128))]

In [37]:
# Ranking the documents 
results = sorted(
    results,
    key=lambda x: x[1],
    reverse=True
)
for document, score in results:
    print(f"{score:.3f} | {document}")

0.707 | Employees receive 30 days of annual leave per calendar year.
0.552 | Employees may work remotely up to three days per week.
0.337 | The company reviews salaries every January.
0.326 | Employees can request unpaid leave with manager approval.
0.240 | The company provides occupational health services.


In [38]:
# Top-k retrieval
k = 3
top_results = results[:k]
for document, score in top_results:
    print(f"{score:.3f} | {document}")

0.707 | Employees receive 30 days of annual leave per calendar year.
0.552 | Employees may work remotely up to three days per week.
0.337 | The company reviews salaries every January.


# Problem with this retrieval approach
What happens when there are  1000000 chuncks? 
The approach will be very computationally heavy everytime a user makes a query.

FAISS: Facebook AI Similarity Search
It is a library designed for efficient similarity search over vectors.

So instead of:
query
 ↓
compare against every vector
 ↓
sort

We create a index:
documents
   ↓
embeddings
   ↓
FAISS index

Then:
query
   ↓
embedding
   ↓
FAISS
   ↓
nearest vectors

Building a mini FAISS
documents > embeddings > FAISS index > user questions > query embeddings > FAISS similarity search > top k chuncks

In [39]:
import faiss
dimension = document_embeddings.shape[1]
# With normalized vectors, inner product i.e., IP is equivalent to cosine similarity. 
index = faiss.IndexFlatIP(dimension)

In [40]:
import numpy as np
document_embeddings = model.encode(
    documents,
    normalize_embeddings=True
)
index = faiss.IndexFlatIP(
    document_embeddings.shape[1]
)
index.add(
    np.array(document_embeddings)
)

In [41]:
# Search the index for query
query_embedding = model.encode(
    [query],
    normalize_embeddings=True
)
scores, indices = index.search(
    np.array(query_embedding),
    k=3
)

In [42]:
indices

array([[0, 1, 4]])

In [43]:
scores

array([[0.7066408 , 0.5517914 , 0.33673126]], dtype=float32)

In [44]:
for score, index_id in zip(scores[0], indices[0]):
    print(f"{score:.3f} | {documents[index_id]}")

0.707 | Employees receive 30 days of annual leave per calendar year.
0.552 | Employees may work remotely up to three days per week.
0.337 | The company reviews salaries every January.


# Chunking: The Hidden Engine of RAG
Chucking is as  important as embedding. A poor chunking can ruin retrieval even with excellent embeddings. 

Why Chunking exits?
A chunk is a small unit of text stored and searched independently. 
Too large chunks may result in too much noise where as small chunks results in missing information.

Chuck Overlap:
Suppose:
Employees receive 30 days of annual leave.
Unused leave can be transferred to the next year.

Overlap:
Chunk 1:
Employees receive 30 days of annual leave.
Chunk 2:
30 days of annual leave.
Unused leave can be transferred...

In [51]:
from src.chunker import character_chunker
text = """
Employees receive 30 days of annual leave.
Employees may work remotely.
Employees receive occupational healthcare.
""" * 20

chunks = character_chunker(text)
print(len(chunks))
print(chunks[0])

10

Employees receive 30 days of annual leave.
Employees may work remotely.
Employees receive occupational healthcare.

Employees receive 30 days of annual leave.
Employees may work remotely.
Employees receive occupational healthcare.

Employees receive 30 days of annual leave.
Employees may work remot


In [52]:
import re
def sentence_split(text):
    return re.split(r'[.!?]', text)


text = """
Employees receive annual leave.
Employees can work remotely.
Managers approve leave.
"""

sentences = sentence_split(text)    